In [5]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- PASSO A: Carichiamo e Uniamo i VOLI (Verticale) ---
# Sostituisci con i tuoi nomi file reali
file_voli_24 = "Dati aeroporto/2024 estrazione dati di traffico BLQ.xlsx" # o .xlsx
file_voli_25 = "Dati aeroporto/2025 estrazione dati di traffico BLQ_fino 1007.xlsx"

In [6]:
df_v24 = pd.read_excel(file_voli_24)
df_v25 = pd.read_excel(file_voli_25)

In [7]:
# Incolliamo il 2025 sotto il 2024
df_voli_tot = pd.concat([df_v24, df_v25], ignore_index=True)

In [8]:
# ==============================================================================
# INIZIO BLOCCO: DATASET SCIENTIFICO PULITO (Data | Voli | Agenti)
# ==============================================================================

# 1. TRASFORMAZIONE VOLI: DA ELENCO A CONTEGGIO ORARIO
print("--- FASE 1: Calcolo traffico aereo orario ---")

df_voli_tot['BLOCK_TIME'] = pd.to_datetime(df_voli_tot['BLOCK_TIME'])
# Creiamo il riferimento orario
df_voli_tot['Gancio_Ora'] = df_voli_tot['BLOCK_TIME'].dt.floor('h')

# CONTEGGIO: 1 riga = 1 ora
voli_per_ora = df_voli_tot.groupby('Gancio_Ora').size().reset_index(name='Numero_Voli')

# 2. CREAZIONE TIMELINE COMPLETA (Il Master Temporale)
print("--- FASE 2: Generazione Timeline (24h/24h) ---")
start_date = df_voli_tot['Gancio_Ora'].min()
end_date = df_voli_tot['Gancio_Ora'].max()

# Generiamo tutte le ore (incluse le notti vuote)
timeline_completa = pd.date_range(start=start_date, end=end_date, freq='h', name='Gancio_Ora')
df_master = pd.DataFrame(timeline_completa)

# Uniamo i voli (mettiamo 0 dove non ce ne sono)
df_master = pd.merge(df_master, voli_per_ora, on='Gancio_Ora', how='left')
df_master['Numero_Voli'] = df_master['Numero_Voli'].fillna(0).astype(int)

# Gancio di servizio per i dati giornalieri (verrà rimosso alla fine)
df_master['Gancio_Giorno'] = df_master['Gancio_Ora'].dt.normalize()

# 3. LISTA AGENTI
lista_agenti = [
    {'nome_colonna': 'C6H6', 'path_24': 'Dati aria/C6H6_2024.csv', 'path_25': 'Dati aria/C6H6_2025.csv', 'col_valore_orig': 'PORTA SAN FELICE'},
    {'nome_colonna': 'PM10', 'path_24': 'Dati aria/PM10_2024.csv', 'path_25': 'Dati aria/PM10_2025.csv', 'col_valore_orig': 'PORTA SAN FELICE'},
    {'nome_colonna': 'CO',   'path_24': 'Dati aria/CO_2024.csv',   'path_25': 'Dati aria/CO_2025.csv',   'col_valore_orig': 'PORTA SAN FELICE'},
    {'nome_colonna': 'NO2',  'path_24': 'Dati aria/NO2_2024.csv',  'path_25': 'Dati aria/NO2_2025.csv',  'col_valore_orig': 'PORTA SAN FELICE'},
    {'nome_colonna': 'PM25', 'path_24': 'Dati aria/PM25_2024.csv', 'path_25': 'Dati aria/PM25_2025.csv', 'col_valore_orig': 'PORTA SAN FELICE'}
]

# 4. CICLO DI UNIONE DATI ARIA
print("--- FASE 3: Integrazione Dati Ambientali ---")

for agente in lista_agenti:
    nome = agente['nome_colonna']
    print(f" -> Elaborazione: {nome}")
    try:
        # A. Caricamento e Pulizia
        df_agente = pd.concat([
            pd.read_csv(agente['path_24'], encoding='latin1', sep=','),
            pd.read_csv(agente['path_25'], encoding='latin1', sep=',')
        ], ignore_index=True)

        df_agente.columns = df_agente.columns.str.strip().str.lower()
        if 'data' in df_agente.columns:
            df_agente['data'] = pd.to_datetime(df_agente['data'], dayfirst=True, errors='coerce')
            df_agente = df_agente.dropna(subset=['data'])
        else:
            continue # Salta se non trova la data

        col_val = agente['col_valore_orig'].strip().lower()
        if col_val in df_agente.columns:
            df_agente = df_agente.rename(columns={col_val: nome})
        else:
            continue

        # B. Creazione Ganci
        df_agente['Gancio_Ora'] = df_agente['data'].dt.floor('h')
        df_agente['Gancio_Giorno'] = df_agente['data'].dt.normalize()

        # C. Decisione Tipo (Orario o Giornaliero)
        ore_uniche = df_agente['data'].dt.hour.unique()
        is_daily = (len(ore_uniche) == 1 and ore_uniche[0] == 0)
        chiave = 'Gancio_Giorno' if is_daily else 'Gancio_Ora'

        # D. Aggregazione e Merge
        df_agente_clean = df_agente.groupby(chiave)[nome].mean().reset_index()
        df_master = pd.merge(df_master, df_agente_clean, on=chiave, how='left')

    except Exception as e:
        print(f"❌ ERRORE su {nome}: {e}")

# ==============================================================================
# 5. PULIZIA FINALE E LAYOUT (Quello che hai chiesto)
# ==============================================================================
print("\n--- FASE 4: Pulizia Finale Layout ---")

# 1. Definiamo l'ordine esatto delle colonne che vuoi
colonne_finali = ['Gancio_Ora', 'Numero_Voli'] + [ag['nome_colonna'] for ag in lista_agenti]

# 2. Selezioniamo solo quelle colonne (filtrando quelle che magari non si sono caricate per errore)
colonne_esistenti = [c for c in colonne_finali if c in df_master.columns]
df_finale_pulito = df_master[colonne_esistenti].copy()

# 3. Rinominiamo la colonna tecnica "Gancio_Ora" in "Data"
df_finale_pulito = df_finale_pulito.rename(columns={'Gancio_Ora': 'Data'})

# 4. Mostra e Salva
print(f"✅ DATASET PULITO PRONTO! Dimensioni: {df_finale_pulito.shape}")
display(df_finale_pulito.head(10))

df_finale_pulito.to_excel("Dataset_Scientifico_Pulito.xlsx", index=False)

--- FASE 1: Calcolo traffico aereo orario ---
--- FASE 2: Generazione Timeline (24h/24h) ---
--- FASE 3: Integrazione Dati Ambientali ---
 -> Elaborazione: C6H6
 -> Elaborazione: PM10
 -> Elaborazione: CO
 -> Elaborazione: NO2
 -> Elaborazione: PM25

--- FASE 4: Pulizia Finale Layout ---
✅ DATASET PULITO PRONTO! Dimensioni: (13368, 7)


,Data,Numero_Voli,C6H6,PM10,CO,NO2,PM25
0,2024-01-01 00:00:00,1,NaN,NaN,NaN,NaN,NaN
1,2024-01-01 01:00:00,1,NaN,NaN,NaN,NaN,NaN
2,2024-01-01 02:00:00,0,NaN,NaN,NaN,NaN,NaN
3,2024-01-01 03:00:00,0,NaN,NaN,NaN,NaN,NaN
4,2024-01-01 04:00:00,0,NaN,NaN,NaN,NaN,NaN
5,2024-01-01 05:00:00,2,NaN,NaN,NaN,NaN,NaN
6,2024-01-01 06:00:00,8,NaN,NaN,NaN,NaN,NaN
7,2024-01-01 07:00:00,10,NaN,NaN,NaN,NaN,NaN
8,2024-01-01 08:00:00,7,NaN,NaN,NaN,NaN,NaN
9,2024-01-01 09:00:00,6,NaN,NaN,NaN,NaN,NaN
